[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 01](README.md)

# Amdahl, Gustafson y escalabilidad

**Tema:** 01 · **Sesiones:** 4, 6 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Por qué una aceleración observada debe interpretarse junto con eficiencia, tamaño y modelo de escalado?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Una aceleración aislada puede parecer buena y aun ocultar baja eficiencia. Este recorrido conecta los límites analíticos con una serie de tiempos observados.

**Prerrequisitos.**

- Aritmética, funciones y lectura de gráficas.
- Python básico para modificar parámetros y ejecutar aserciones.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Calcular límites de Amdahl y Gustafson.
- Distinguir escalado fuerte y débil.
- Reportar eficiencia y overhead con una línea base estable.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Amdahl mantiene el tamaño fijo y hace visible la fracción serial.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Gustafson razona sobre problemas cuyo trabajo paralelo crece con los recursos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

La eficiencia Sp/p cae por serialización, comunicación, desbalance y costos del runtime.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- trabajo T₁ — costo total de todas las operaciones
- span T∞ — costo del camino dependiente más largo
- eficiencia — aceleración dividida entre recursos
- intensidad aritmética — FLOP realizados por byte movido


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Escalabilidad

![Curvas ideal, limitada y observada](../../images/escalabilidad.svg)

**Cómo leerlo.** Compara la pendiente de cada curva y pregunta siempre si el tamaño es fijo. La distancia frente a la línea ideal se interpreta con eficiencia y overhead.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "01"
NOTEBOOK = "01_fundamentos/02_escalabilidad.ipynb"
assert (ROOT / "curso" / "notebooks" / "01_fundamentos" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Límites analíticos

**Situación.** Se comparan ambos modelos para la misma fracción paralela.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def amdahl(parallel_fraction, p): return 1 / ((1 - parallel_fraction) + parallel_fraction / p)
def gustafson(parallel_fraction, p): return p - (1 - parallel_fraction) * (p - 1)
fraction = 0.95
rows = []
for p in (1, 2, 4, 8, 16, 32):
    rows.append((p, amdahl(fraction, p), gustafson(fraction, p)))
assert rows[0] == (1, 1.0, 1.0)
for p, strong, scaled in rows: print(f"p={p:2} Amdahl={strong:6.2f} Gustafson={scaled:6.2f}")


### Explicación del resultado

Los modelos responden preguntas diferentes; no deben presentarse como predicciones intercambiables.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Métricas observadas

**Situación.** Se calcula aceleración, eficiencia y overhead a partir de tiempos sintéticos trazables.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
serial = 12.0
times = {1: 12.0, 2: 6.4, 4: 3.5, 8: 2.1, 16: 1.65}
for p, elapsed in times.items():
    speedup = serial / elapsed
    efficiency = speedup / p
    overhead = p * elapsed - serial
    assert 0 < efficiency <= 1.000001
    print(f"p={p:2} t={elapsed:4.2f} S={speedup:5.2f} E={efficiency:5.3f} To={overhead:5.2f}")


### Lectura razonada

El tiempo de p=1 de la versión paralela y el serial optimizado son referencias distintas y deben etiquetarse.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Por qué Amdahl y Gustafson pueden producir conclusiones distintas sin que uno de los dos esté equivocado?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Formular por separado una hipótesis fuerte y una débil.
2. Usar al menos cinco repeticiones y reportar dispersión.
3. Identificar el primer punto de saturación y una causa verificable.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Comparar ejecutables con opciones de compilación distintas.
- Usar el mejor tiempo sin justificarlo.
- Omitir tamaño del problema, afinidad o hardware.


## Criterios de aceptación

- Definición explícita de tamaño fijo o trabajo por recurso.
- Aceleración y eficiencia derivadas de los mismos datos.
- Datos crudos y manifiesto de plataforma conservados.


## Síntesis

- La pregunta que debes poder responder es: **¿Por qué una aceleración observada debe interpretarse junto con eficiencia, tamaño y modelo de escalado?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Protocolo experimental](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md#8-niveles-de-evidencia)
- [Planeación](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 01](README.md)
